<a href="https://colab.research.google.com/github/sulucay01/DI725-assignment1/blob/dev/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing

This notebook prepares the raw customer support conversations for downstream modeling.

The preprocessing pipeline includes:
- loading raw train and test datasets
- removing malformed conversations
- cleaning and normalizing text
- standardizing categorical variables
- creating simple text-based features
- encoding labels for the training set
- creating a stratified validation split
- saving processed datasets

In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

os.makedirs("data/processed", exist_ok=True)

## 1. Load Raw Data

In this section, we load the raw train and test datasets and inspect their basic structure.

In [2]:
train = pd.read_csv(
    "https://raw.githubusercontent.com/sulucay01/DI725-assignment1/main/data/raw/train.csv"
)
test = pd.read_csv(
    "https://raw.githubusercontent.com/sulucay01/DI725-assignment1/main/data/raw/test.csv"
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain columns:")
print(train.columns.tolist())

train.head(3)

Train shape: (970, 11)
Test shape: (30, 11)

Train columns:
['issue_area', 'issue_category', 'issue_sub_category', 'issue_category_sub_category', 'customer_sentiment', 'product_category', 'product_sub_category', 'issue_complexity', 'agent_experience_level', 'agent_experience_level_desc', 'conversation']


,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or email address during login,Mobile Number and Email Verification -> Verification requirement for mobile number or email address during login,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, possess solid troubleshooting skills, and seek guidance from more experienced team members when needed.","Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG),..."
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked to ship the item,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, possess solid troubleshooting skills, and seek guidance from more experienced team members when needed.",Agent: Thank you for calling BrownBox customer support. My name is Alex. How may I assist you today?\n\nCustomer: Hi Alex. I recently received an email from BrownBox requesting me to ship back the...
2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to click the 'Cancel' button,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, excel in de-escalation, and possess the ability to empathize with customers, providing them with effective solutions and support.","Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How may I assist you today?\n\nCustomer: Hi Sarah, I am calling because I am unable to click the 'Cancel' button for my Ju..."


## 2. Define Preprocessing Functions

To make the preprocessing pipeline reusable and easier to maintain, we define helper functions for:
- removing malformed rows
- cleaning conversation text
- normalizing categorical variables
- generating simple text-derived features
- applying the full preprocessing pipeline

In [3]:
def remove_invalid_conversations(df):
    df = df.copy()

    mask_no_customer = (
        df["conversation"].str.contains(r"\bAgent\s*:", regex=True, na=False, case=False)
        & ~df["conversation"].str.contains(r"\bCustomer\s*:", regex=True, na=False, case=False)
    )

    removed_count = mask_no_customer.sum()
    df = df.loc[~mask_no_customer].reset_index(drop=True)

    return df, removed_count

In [4]:
def clean_conversation(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Convert escaped line breaks first
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")

    # Then normalize real line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace speaker tags
    text = re.sub(r"\bAgent\s*:", "[AGENT]", text, flags=re.IGNORECASE)
    text = re.sub(r"\bCustomer\s*:", "[CUSTOMER]", text, flags=re.IGNORECASE)

    # Mask patterns
    text = re.sub(
        r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
        "[EMAIL]",
        text
    )
    text = re.sub(
        r"(?<!\w)(?:\+\d{1,3}\s*)?(?:\(\d{3}\)|\d{3})[-.\s]\d{3}[-.\s]\d{4}(?!\w)",
        "[PHONE]",
        text
    )
    text = re.sub(r"\b\d{5,}\b", "[NUMBER]", text)
    text = re.sub(
        r"\b[A-Z]{2,}[A-Z0-9-]*\d+[A-Z0-9-]*\b|\b[A-Z]{2,}-\d+[A-Z0-9-]*\b",
        "[ID]",
        text
    )

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()

In [5]:
def normalize_categorical_columns(df):
    """
    Standardize categorical columns by filling missing values
    and trimming extra whitespace.
    """
    df = df.copy()

    for col in ["issue_area", "issue_category"]:
        if col in df.columns:
            df[col] = df[col].fillna("unknown").astype(str).str.strip()

    return df

In [6]:
def add_text_features(df):
    """
    Create simple text-derived features from the cleaned conversation text.
    """
    df = df.copy()

    df["char_len"] = df["clean_text"].astype(str).apply(len)
    df["word_len"] = df["clean_text"].astype(str).apply(lambda x: len(x.split()))
    df["line_count"] = df["clean_text"].astype(str).apply(lambda x: x.count("\n") + 1)

    return df

In [7]:
def preprocess_dataframe(df, dataset_name="dataset"):
    """
    Apply the full preprocessing pipeline to a dataframe.
    """
    df = df.copy()

    print(f"\nPreprocessing {dataset_name}...")
    print(f"Initial shape: {df.shape}")

    # Remove malformed rows
    df, removed_invalid = remove_invalid_conversations(df)
    print(f"Removed malformed rows: {removed_invalid}")
    print(f"Shape after malformed-row removal: {df.shape}")

    # Remove duplicate raw conversations
    before_raw_dedup = len(df)
    df = df.drop_duplicates(subset=["conversation"]).reset_index(drop=True)
    print(f"Removed duplicate raw conversations: {before_raw_dedup - len(df)}")
    print(f"Shape after raw deduplication: {df.shape}")

    # Normalize categorical columns
    df = normalize_categorical_columns(df)

    # Clean text
    df["clean_text"] = df["conversation"].apply(clean_conversation)

    # Remove duplicate cleaned conversations
    before_clean_dedup = len(df)
    df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)
    print(f"Removed duplicate cleaned conversations: {before_clean_dedup - len(df)}")
    print(f"Shape after cleaned-text deduplication: {df.shape}")

    # Add lightweight text features
    df = add_text_features(df)

    return df

## 3. Preprocess Training and Test Data

We now apply the same preprocessing pipeline to both the training and test sets.

The pipeline:
- removes malformed rows
- removes duplicates
- normalizes categorical fields
- cleans conversation text
- adds simple text-based features

In [8]:
train_processed = preprocess_dataframe(train, dataset_name="train")
test_processed = preprocess_dataframe(test, dataset_name="test")

print("\nProcessed train shape:", train_processed.shape)
print("Processed test shape:", test_processed.shape)


Preprocessing train...
Initial shape: (970, 11)
Removed malformed rows: 3
Shape after malformed-row removal: (967, 11)
Removed duplicate raw conversations: 0
Shape after raw deduplication: (967, 11)
Removed duplicate cleaned conversations: 0
Shape after cleaned-text deduplication: (967, 12)

Preprocessing test...
Initial shape: (30, 11)
Removed malformed rows: 0
Shape after malformed-row removal: (30, 11)
Removed duplicate raw conversations: 0
Shape after raw deduplication: (30, 11)
Removed duplicate cleaned conversations: 0
Shape after cleaned-text deduplication: (30, 12)

Processed train shape: (967, 15)
Processed test shape: (30, 15)


In [9]:
print("Null clean_text:", train_processed["clean_text"].isna().sum())
print("Empty clean_text:", (train_processed["clean_text"].str.strip() == "").sum())
print("Rows equal to 'nan':", (train_processed["clean_text"] == "nan").sum())

print("Rows with [CUSTOMER]:", train_processed["clean_text"].str.contains(r"\[CUSTOMER\]", regex=True, na=False).sum())
print("Rows with [AGENT]:", train_processed["clean_text"].str.contains(r"\[AGENT\]", regex=True, na=False).sum())

print("Rows still containing Customer:", train_processed["clean_text"].str.contains(r"\bCustomer\s*:", regex=True, na=False).sum())
print("Rows still containing Agent:", train_processed["clean_text"].str.contains(r"\bAgent\s*:", regex=True, na=False).sum())

print(train_processed["clean_text"].str.len().describe())
print(train_processed[["conversation", "clean_text"]].head(10).to_string(index=False))

Null clean_text: 0
Empty clean_text: 0
Rows equal to 'nan': 0
Rows with [CUSTOMER]: 967
Rows with [AGENT]: 967
Rows still containing Customer: 0
Rows still containing Agent: 0
count     967.000000
mean     2128.441572
std       551.057170
min       745.000000
25%      1760.000000
50%      2054.000000
75%      2423.000000
max      5653.000000
Name: clean_text, dtype: float64
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [10]:
for i in range(3):
    print("="*80)
    print("RAW:")
    print(repr(train.loc[i, "conversation"]))
    print("\nCLEAN:")
    print(repr(train_processed.loc[i, "clean_text"]))

RAW:
"Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email verification. Can you help me with that?\n\nAgent: Sure, I can assist you with that. May I know your registered mobile number or email address, please?\n\nCustomer: My registered mobile number is +1 123-456-7890.\n\nAgent: Thank you. Let me check that for you. I'm sorry to inform you that we don't have this number on our records. Can you please confirm if this is the correct number?\n\nCustomer: Oh, I'm sorry. I might have registered with a different number. Can you please check with my email address instead? It's johndoe@email.com.\n\nAgent: Sure, let me check that for you. (After a few moments) I see that we have your email address on our records. We'll be sending you a verification code shortly. Please check your ema

In [11]:
txt = train_processed.loc[0, "clean_text"]

print("PRINT OUTPUT:")
print(txt)

print("\nREPR OUTPUT:")
print(repr(txt))

PRINT OUTPUT:
[AGENT] Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?
[CUSTOMER] Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email verification. Can you help me with that?
[AGENT] Sure, I can assist you with that. May I know your registered mobile number or email address, please?
[CUSTOMER] My registered mobile number is [PHONE].
[AGENT] Thank you. Let me check that for you. I'm sorry to inform you that we don't have this number on our records. Can you please confirm if this is the correct number?
[CUSTOMER] Oh, I'm sorry. I might have registered with a different number. Can you please check with my email address instead? It's [EMAIL].
[AGENT] Sure, let me check that for you. (After a few moments) I see that we have your email address on our records. We'll be sending you a verification code shortly. Please check your email and let me know on

In [12]:
for i in range(3):
    txt = train_processed.loc[i, "clean_text"]
    print("="*80)
    print("ROW", i)
    print("real newline present:", "\n" in txt)
    print("literal \\n present:", "\\n" in txt)
    print("\nVISIBLE TEXT:")
    print(txt[:500])

ROW 0
real newline present: True
literal \n present: False

VISIBLE TEXT:
[AGENT] Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?
[CUSTOMER] Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'm unable to proceed as it's asking for mobile number or email verification. Can you help me with that?
[AGENT] Sure, I can assist you with that. May I know your registered mobile number or email address, please?
[CUSTOMER] My registered mobile number is [PHONE].
[AGENT] Thank you. Let me check tha
ROW 1
real newline present: True
literal \n present: False

VISIBLE TEXT:
[AGENT] Thank you for calling BrownBox customer support. My name is Alex. How may I assist you today?
[CUSTOMER] Hi Alex. I recently received an email from BrownBox requesting me to ship back the computer monitor I purchased last week. Can you please tell me why I am being asked to ship it back?
[AGENT] Sure, I'll be happy to help you with that. May

## 4. Encode Target Labels

The target variable `customer_sentiment` is converted into numeric labels for model training.

We also save the label mapping so that the relationship between class names and encoded values remains explicit and reproducible.

In [13]:
label_encoder = LabelEncoder()
train_processed["label"] = label_encoder.fit_transform(train_processed["customer_sentiment"])
test_processed["label"] = label_encoder.transform(test_processed["customer_sentiment"])

label_mapping = {
    class_name: int(idx) for idx, class_name in enumerate(label_encoder.classes_)
}

inverse_label_mapping = {
    int(idx): class_name for idx, class_name in enumerate(label_encoder.classes_)
}

print("Label mapping:")
print(label_mapping)

print("\nCustomer sentiment distribution:")
print(train_processed["customer_sentiment"].value_counts())

print("\nEncoded label distribution:")
print(train_processed["label"].value_counts())

Label mapping:
{'negative': 0, 'neutral': 1, 'positive': 2}

Customer sentiment distribution:
customer_sentiment
neutral     542
negative    408
positive     17
Name: count, dtype: int64

Encoded label distribution:
label
1    542
0    408
2     17
Name: count, dtype: int64


## 5. Create Train/Validation Split

To evaluate models fairly, the processed training data is split into train and validation sets using stratification on the target label.
This helps preserve the class distribution in both subsets.

In [14]:
train_split, val_split = train_test_split(
    train_processed,
    test_size=0.2,
    stratify=train_processed["label"],
    random_state=42
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print("Train split shape:", train_split.shape)
print("Validation split shape:", val_split.shape)

print("\nTrain label distribution:")
print(train_split["customer_sentiment"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_split["customer_sentiment"].value_counts(normalize=True))

Train split shape: (773, 16)
Validation split shape: (194, 16)

Train label distribution:
customer_sentiment
neutral     0.560155
negative    0.421734
positive    0.018111
Name: proportion, dtype: float64

Validation label distribution:
customer_sentiment
neutral     0.561856
negative    0.422680
positive    0.015464
Name: proportion, dtype: float64


## 6. Save Processed Datasets

Finally, we save the processed train, validation, and test datasets, along with the label mapping, for downstream modeling experiments.

In [15]:
train_split.to_csv("data/processed/train_processed.csv", index=False)
val_split.to_csv("data/processed/val_processed.csv", index=False)
test_processed.to_csv("data/processed/test_processed.csv", index=False)

with open("data/processed/label_mapping.json", "w") as f:
    json.dump(
        {
            "label_to_id": label_mapping,
            "id_to_label": inverse_label_mapping
        },
        f,
        indent=4
    )

print("Saved files:")
print("- data/processed/train_processed.csv")
print("- data/processed/val_processed.csv")
print("- data/processed/test_processed.csv")
print("- data/processed/label_mapping.json")

Saved files:
- data/processed/train_processed.csv
- data/processed/val_processed.csv
- data/processed/test_processed.csv
- data/processed/label_mapping.json


## 7. Sanity Checks

As a final check, we inspect a few cleaned examples and verify the final column structure of the processed training set.

In [16]:
for i in range(3):
    print(f"\n===== SAMPLE {i} =====")
    print(train_split.loc[i, "clean_text"][:1000])

print("\nFinal columns:")
print(train_split.columns.tolist())


===== SAMPLE 0 =====
[AGENT] Hello, thank you for contacting BrownBox customer support. My name is Jane, how may I assist you today?
[CUSTOMER] Hi Jane, I placed an order for a power bank a week ago, and I still haven't received it. Can you please tell me when I can expect the delivery?
[AGENT] I apologize for the delay, and I understand your concern. May I please have your order number and registered email address so that I can check the details and assist you further?
[CUSTOMER] Sure, my order number is [ID], and my registered email address is [EMAIL].
[AGENT] Thank you, Jane. Let me quickly check the status of your order and see what we can do to resolve this issue.
(Customer is put on hold for a minute)
[AGENT] Thank you for your patience, Jane. I have checked the details, and I see that the power bank is currently out of stock. We are expecting a new shipment in a few days, and your order will be shipped as soon as the stock is available.
[CUSTOMER] What? Why didn't anyone inform